## 模型初始化

In [1]:
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

load_dotenv(override=True)
ARK_API_KEY = os.getenv("ARK_API_KEY")
ARK_MODEL_NAME = os.getenv("ARK_MODEL_NAME")
ARK_BASE_URL = os.getenv("ARK_BASE_URL")
model = ChatOpenAI(
    model= ARK_MODEL_NAME,
    api_key= ARK_API_KEY,
    base_url= ARK_BASE_URL
)

c:\Users\XSH\.conda\envs\langchain1.2\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## 工具定义

In [3]:
from langchain.tools import tool
from typing import Optional
import math

@tool
def calculator(expression: str) -> str:
    """计算数学表达式。输入应该是一个有效的Python数学表达式，例如 '2 + 3 * 4' 或 'sqrt(16)'。
    
    Args:
        expression: 要计算的数学表达式字符串
    """
    try:
        # 安全的环境中计算
        allowed_names = {
            k: v for k, v in math.__dict__.items() if not k.startswith("_")
        }
        allowed_names.update({"abs": abs, "round": round})
        
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"计算结果: {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"

@tool
def string_processor(text: str, operation: str = "reverse") -> str:
    """处理字符串。支持的操作：
    - reverse: 反转字符串
    - upper: 转大写
    - lower: 转小写
    - length: 返回长度
    - word_count: 统计单词数
    
    Args:
        text: 要处理的文本
        operation: 操作类型 (reverse/upper/lower/length/word_count)
    """
    operations = {
        "reverse": lambda s: s[::-1],
        "upper": lambda s: s.upper(),
        "lower": lambda s: s.lower(),
        "length": lambda s: str(len(s)),
        "word_count": lambda s: str(len(s.split())),
    }
    
    if operation not in operations:
        return f"不支持的操作: {operation}。支持: {', '.join(operations.keys())}"
    
    result = operations[operation](text)
    return f"结果: {result}"

import random

@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """查询指定城市的天气信息（模拟数据）。
    
    Args:
        city: 城市名称
        unit: 温度单位，celsius 或 fahrenheit
    """
    # 模拟天气数据
    conditions = ["晴天", "多云", "小雨", "阴天", "雷阵雨"]
    temp_c = random.randint(10, 35)
    
    if unit == "fahrenheit":
        temp = temp_c * 9/5 + 32
        unit_symbol = "°F"
    else:
        temp = temp_c
        unit_symbol = "°C"
    
    condition = random.choice(conditions)
    humidity = random.randint(30, 90)
    
    return (
        f"{city}天气：{condition}，"
        f"温度 {temp}{unit_symbol}，"
        f"湿度 {humidity}%"
    )


@tool
def query_database(table: str, query_type: str = "count", limit: int = 5) -> str:
    """模拟数据库查询工具。
    
    Args:
        table: 表名 (users/orders/products)
        query_type: 查询类型 (count/select/schema)
        limit: 返回结果数量限制
    """
    # 模拟数据库
    mock_db = {
        "users": [
            {"id": 1, "name": "Alice", "age": 30},
            {"id": 2, "name": "Bob", "age": 25},
            {"id": 3, "name": "Charlie", "age": 35},
        ],
        "orders": [
            {"id": 101, "user_id": 1, "amount": 99.9},
            {"id": 102, "user_id": 2, "amount": 150.0},
        ],
        "products": [
            {"id": 1, "name": "Laptop", "price": 999},
            {"id": 2, "name": "Phone", "price": 599},
        ],
    }
    
    if table not in mock_db:
        return f"表 '{table}' 不存在。可用表: {', '.join(mock_db.keys())}"
    
    data = mock_db[table]
    
    if query_type == "count":
        return f"表 {table} 共有 {len(data)} 条记录"
    elif query_type == "select":
        results = data[:limit]
        return f"查询结果 ({len(results)} 条):\n" + "\n".join(str(r) for r in results)
    elif query_type == "schema":
        if data:
            fields = list(data[0].keys())
            return f"表 {table} 的字段: {', '.join(fields)}"
        return f"表 {table} 为空"
    else:
        return f"不支持的查询类型: {query_type}"

@tool
def send_email(to: str, subject: str, body: str, cc: Optional[str] = None) -> str:
    """发送邮件（模拟）。
    
    Args:
        to: 收件人邮箱
        subject: 邮件主题
        body: 邮件正文
        cc: 抄送邮箱（可选）
    """
    # 简单的邮箱格式验证
    if "@" not in to:
        return f"错误: 无效的邮箱地址 '{to}'"
    
    result = f"✅ 邮件已发送\n收件人: {to}\n主题: {subject}\n正文长度: {len(body)} 字符"
    if cc:
        result += f"\n抄送: {cc}"
    
    return result

@tool
def file_operations(action: str, filename: str, content: str = "") -> str:
    """模拟文件操作。
    
    Args:
        action: 操作类型 (read/write/append/delete/list)
        filename: 文件名
        content: 写入内容（write/append时使用）
    """
    # 模拟文件系统
    mock_fs = {
        "test.txt": "这是测试文件内容",
        "data.csv": "name,age\nAlice,30\nBob,25",
    }
    
    if action == "read":
        if filename in mock_fs:
            return f"文件 '{filename}' 内容:\n{mock_fs[filename]}"
        return f"文件 '{filename}' 不存在"
    
    elif action == "write":
        mock_fs[filename] = content
        return f"已写入文件 '{filename}'，内容长度: {len(content)}"
    
    elif action == "append":
        if filename in mock_fs:
            mock_fs[filename] += "\n" + content
            return f"已追加内容到 '{filename}'"
        mock_fs[filename] = content
        return f"文件不存在，已创建 '{filename}'"
    
    elif action == "delete":
        if filename in mock_fs:
            del mock_fs[filename]
            return f"已删除文件 '{filename}'"
        return f"文件 '{filename}' 不存在"
    
    elif action == "list":
        files = list(mock_fs.keys())
        return f"文件列表: {', '.join(files) if files else '(空)'}"
    
    return f"不支持的操作: {action}"

calculator — 计算器
作用：计算数学表达式，测试数值计算和基础工具调用。
输入：一个数学表达式字符串
示例：2 + 3 * 4 得 14，sqrt(16) 得 4.0

string_processor — 字符串处理
作用：对文本做各种操作，测试多参数和枚举选项的参数识别。
输入：一段文本 + 操作类型
支持操作：反转、转大写、转小写、字符长度、单词数

get_weather — 天气查询
作用：查询城市天气（模拟数据），测试带单位转换的参数处理。
输入：城市名 + 温度单位（摄氏/华氏）
返回：天气状况、温度、湿度

query_database — 数据库查询
作用：模拟数据库查询，测试结构化数据和多查询类型的工具调用。
输入：表名 + 查询类型 + 数量限制
支持查询：统计记录数、查询记录、查看表结构
可用表：users、orders、products

send_email — 邮件发送
作用：发送邮件（模拟，不真发），测试带可选参数的工具调用和参数校验。
输入：收件人、主题、正文、抄送（可选）
特点：校验邮箱格式，非法地址返回错误

file_operations — 文件操作
作用：模拟文件系统读写删查，测试有状态操作（写入后能读到）。
输入：操作类型 + 文件名 + 内容（写入时用）
支持操作：读取、写入、追加、删除、列出文件

简单写一个测试工具

## Agent实例化

In [17]:

class AiAssistant:
    def __init__(self):
        self.model = model
        self.tools = [calculator,string_processor,get_weather,query_database,send_email,file_operations]
        system_prompt =  """你是一个可以调用工具的测试助手。

        可用工具：

        - calculator：计算数学表达式，如 "2 + 3 * 4" 或 "sqrt(16)"。
        - string_processor：处理字符串，操作包括 reverse、upper、lower、length、word_count。
        - get_weather：查询城市天气（模拟），参数为城市名和温度单位。
        - query_database：模拟数据库查询，参数为表名、查询类型（count/select/schema）和数量限制，可用表为 users、orders、products。
        - send_email：发送邮件（模拟），参数为收件人、主题、正文，可选抄送。
        - file_operations：模拟文件操作，操作包括 read、write、append、delete、list。

        规则：

        - 按需选择合适的工具，不要凭空回答。
        - 参数无效或工具报错时，说明原因，不要编造结果。
        - 需要多步时依次调用多个工具。
        - 完成后简洁汇报结果。
        """

        self.agent =  create_agent(
            model=self.model,
            tools=self.tools,
            system_prompt=system_prompt
        )

        # 对话记忆管理
        self.messages = []

    def chat(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})
        result = self.agent.invoke({"messages": self.messages})

        # 保存完整历史，保证多轮上下文
        self.messages = result["messages"]

        # 取最后一条 AI 消息
        last = result["messages"][-1]
        if getattr(last, "content", None):
            return last.content
        return "抱歉，我没法处理您的请求"
    def reset(self):
        self.messages = []
    

## 主程序

In [18]:
def main():
    assistant = AiAssistant()
    print("="*50)
    print("您好，我是一个多功能的人工智障,我可以实现：\n")
    print("- calculator：计算数学表达式，如 2 + 3 * 4 或 sqrt(16).\n",
"- string_processor：处理字符串，操作包括 reverse、upper、lower、length、word_count。\n",
"- get_weather：查询城市天气（模拟），参数为城市名和温度单位。\n",
"- query_database：模拟数据库查询，参数为表名、查询类型（count/select/schema）和数量限制，可用表为 users、orders、products。\n",
"- send_email：发送邮件（模拟），参数为收件人、主题、正文，可选抄送。\n",
"- file_operations：模拟文件操作，操作包括 read、write、append、delete、list。\n")

    while(True):
        user_input = input("\n用户输入：\n")
        if user_input.lower() == "quit":
            break;
        if user_input.lower() == "reset":
            assistant.reset()
            print("对话已重置")
        response = assistant.chat(user_input);
        print("\n智能助手\n",response)
    print("对话结束，期待下次与您见面：）")
if __name__ == "__main__":
    main()

您好，我是一个多功能的人工智障,我可以实现：

- calculator：计算数学表达式，如 2 + 3 * 4 或 sqrt(16).
 - string_processor：处理字符串，操作包括 reverse、upper、lower、length、word_count。
 - get_weather：查询城市天气（模拟），参数为城市名和温度单位。
 - query_database：模拟数据库查询，参数为表名、查询类型（count/select/schema）和数量限制，可用表为 users、orders、products。
 - send_email：发送邮件（模拟），参数为收件人、主题、正文，可选抄送。
 - file_operations：模拟文件操作，操作包括 read、write、append、delete、list。


智能助手
 你好！我是一个**测试助手**，可以通过调用工具来帮助你完成各种任务。

我可以使用的工具包括：

1. **计算器**：计算数学表达式，比如 `2 + 3 * 4`、`sqrt(16)` 等
2. **字符串处理**：反转、大小写转换、统计长度和单词数
3. **天气查询**：查询指定城市的天气信息（模拟数据）
4. **数据库查询**：查询 users、orders、products 三张表的数据（模拟）
5. **发送邮件**：模拟发送邮件
6. **文件操作**：读取、写入、追加、删除文件，或列出文件（模拟）

有什么我可以帮你的吗？比如：
- "帮我算一下 128 * 365 是多少"
- "查一下北京的天气"
- "把这句话反转一下"
- "查一下用户表有多少条数据"

请告诉我你的需求！😊

智能助手
 已经查到杭州今天的天气了：

☀️ **杭州天气**
- 天气：晴天
- 温度：26°C
- 湿度：47%

不过在发送邮件之前，我还需要知道你的**邮箱地址**。请告诉我你要发送到哪个邮箱，我就可以帮你把天气信息发过去了。📧

智能助手
 邮件已成功发送！✅

📧 **发送详情**
- 收件人：123@123.com
- 主题：杭州今日天气信息
- 内容：包含杭州今天晴天、26°C、湿度 47% 的天气信息

请查收邮件，如果还需要其他帮助，随时告诉我！😊

智能助手
 你问的第一个问题是：**"请问你

我还奇怪大模型怎么知道我进行了重置，原来是判断条件之后的执行没写完，把reset也传给了大模型,修改一下，同时让我试试看输入功能外的内容

In [20]:
def main():
    assistant = AiAssistant()
    print("="*50)
    print("您好，我是一个多功能的人工智障,我可以实现：\n")
    print("- calculator：计算数学表达式，如 2 + 3 * 4 或 sqrt(16).\n",
"- string_processor：处理字符串，操作包括 reverse、upper、lower、length、word_count。\n",
"- get_weather：查询城市天气（模拟），参数为城市名和温度单位。\n",
"- query_database：模拟数据库查询，参数为表名、查询类型（count/select/schema）和数量限制，可用表为 users、orders、products。\n",
"- send_email：发送邮件（模拟），参数为收件人、主题、正文，可选抄送。\n",
"- file_operations：模拟文件操作，操作包括 read、write、append、delete、list。\n")

    while(True):
        user_input = input("\n用户输入：\n")
        if user_input.lower() == "quit":
            break
        if user_input.lower() == "reset":
            assistant.reset()
            print("对话已重置")
            continue
        print(f"\n{'=' * 10}用户输入：{'=' * 10}\n{user_input}")
        response = assistant.chat(user_input);
        print(f"\n{'=' * 10}n智能助手：{'=' * 10}\n{response}")
    print("对话结束，期待下次与您见面：）")
if __name__ == "__main__":
    main()

您好，我是一个多功能的人工智障,我可以实现：

- calculator：计算数学表达式，如 2 + 3 * 4 或 sqrt(16).
 - string_processor：处理字符串，操作包括 reverse、upper、lower、length、word_count。
 - get_weather：查询城市天气（模拟），参数为城市名和温度单位。
 - query_database：模拟数据库查询，参数为表名、查询类型（count/select/schema）和数量限制，可用表为 users、orders、products。
 - send_email：发送邮件（模拟），参数为收件人、主题、正文，可选抄送。
 - file_operations：模拟文件操作，操作包括 read、write、append、delete、list。


==========用户输入：==========
我的邮箱是123@123.com 接下来的默认邮箱都是这个

==========n智能助手：==========
好的，已记下：您的邮箱是 **123@123.com**，后续涉及发送邮件或需要邮箱地址的操作时，我将默认使用这个地址。

有什么需要我帮忙的吗？

==========用户输入：==========
帮我查询一下杭州的天气，然后将结果汇总发送到我的邮箱中

==========n智能助手：==========
已完成！✅

**查询结果：**
- 🌦️ 天气状况：雷阵雨
- 🌡️ 温度：24°C
- 💧 湿度：49%

**邮件发送：** 已成功发送到 123@123.com，主题为《杭州天气查询结果》。

温馨提示：雷阵雨天气出门记得带伞哦～还有其他需要帮忙的吗？

==========用户输入：==========
rest

==========n智能助手：==========
请问您的意思是？

1. **重置？**—— 如果您是想输入 "reset"，比如重置之前设置的默认邮箱或其他配置，请告诉我具体要重置什么内容。
2. **REST 接口？**—— 如果您想了解 REST API 相关的内容，可以详细说明您的需求。
3. **休息一下？**—— 如果只是随便说说，那没问题，随时待命，有事叫我～ 😊

请补充说明一下您的需求哦。
对话已重置

====

ok，也是简单尝试了一下，把模块进行拆分，同时规范了一下，之前写的那个业务agent看来还是十分粗糙的，最主要的是没写主函数，后续看下fastapi的相关内容，把功能包装成一个服务看看